# Advanced Tutorial Problems with Solutions
## `yield from`, `throw()`, exception routing, return values, and cleanup

This notebook develops advanced generator-delegation problems in a tutorial style.

Instead of presenting one large solution immediately, each problem is divided into small logical steps:

1. establish the behavior we already understand;
2. predict what Python will do;
3. run a focused experiment;
4. explain the result;
5. refine the implementation;
6. verify the final design with assertions.

The main topic is exception injection through delegated generators:

```python
caller -> delegator -> subgenerator
```

We will study how `next()`, `send()`, `throw()`, and `close()` travel through that chain.


## Learning objectives

By the end of the notebook, you should be able to:

- describe precisely where an exception injected with `throw()` is first raised;
- distinguish a recoverable exception command from a terminating exception;
- explain why yielding directly inside an `except` block can create a surprising suspension point;
- retrieve a subgenerator's return value through `yield from`;
- catch an exception in a delegator after a subgenerator declines to handle it;
- reason about cleanup order when `close()` propagates `GeneratorExit`;
- translate low-level exceptions into domain-level exceptions;
- build restartable and checkpointable generator-based services;
- test coroutine protocols without relying only on printed output.


## Important best practices used throughout

- Use custom exception classes for protocol signals.
- Prime a generator before sending a non-`None` value.
- Put cleanup in `finally`.
- Never yield after receiving `GeneratorExit`.
- Return structured data when a generator terminates.
- Use one predictable suspension point per loop when possible.
- Add assertions after demonstrations so examples double as tests.
- Prefer ordinary message objects for routine commands; reserve `throw()` for exceptional control flow.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from numbers import Real
from typing import Any, Callable, Generator


## Small testing helpers

A generator return value is carried by `StopIteration.value`.

The following helper injects an exception that is expected to terminate a generator and returns that final value.


In [2]:
def throw_and_capture(gen: Generator, exc: BaseException) -> Any:
    try:
        gen.throw(exc)
    except StopIteration as stop:
        return stop.value
    raise AssertionError("The generator did not terminate.")


# Problem 1 — Follow an exception through `yield from`

We will begin with the smallest useful experiment.

Our goal is to answer this question:

> When the caller executes `delegator.throw(SomeError())`, where is the exception raised first?

We will record events in a list rather than depending only on printed output.


## Step 1 — Create a recoverable protocol exception

The subgenerator will recognize `RecoverableSignal`, record it, and continue running.


In [3]:
class RecoverableSignal(Exception):
    pass


## Step 2 — Create the subgenerator

Notice that the active `yield` is inside the inner `try`.

That means an exception thrown into the suspended generator can be caught by the corresponding `except`.


In [4]:
def worker(events: list[str]) -> Generator[str, str, None]:
    output = "worker ready"

    while True:
        try:
            value = yield output
        except RecoverableSignal as exc:
            events.append(f"worker caught: {exc}")
            output = f"recovered from {exc}"
        else:
            events.append(f"worker received: {value}")
            output = f"processed {value}"


## Step 3 — Add a delegator

The delegator itself contains no handler for `RecoverableSignal`.

It simply delegates with `yield from`.


In [5]:
def delegator(events: list[str]) -> Generator[str, str, None]:
    events.append("delegator started")
    yield from worker(events)


## Step 4 — Predict the result

Before running the next cell, decide which component should catch the exception:

- the caller;
- the delegator;
- the worker.

Because `yield from` forwards `throw()` to the active subgenerator, the worker should receive it first.


In [6]:
events: list[str] = []
gen = delegator(events)

first = next(gen)
normal = gen.send("alpha")
recovered = gen.throw(RecoverableSignal("temporary problem"))
after_recovery = gen.send("beta")

first, normal, recovered, after_recovery, events


('worker ready',
 'processed alpha',
 'recovered from temporary problem',
 'processed beta',
 ['delegator started',
  'worker received: alpha',
  'worker caught: temporary problem',
  'worker received: beta'])

## Step 5 — Verify the protocol

The `throw()` call itself returns the next value yielded by the worker after handling the exception.


In [7]:
assert first == "worker ready"
assert normal == "processed alpha"
assert recovered == "recovered from temporary problem"
assert after_recovery == "processed beta"

assert events == [
    "delegator started",
    "worker received: alpha",
    "worker caught: temporary problem",
    "worker received: beta",
]

gen.close()


## What happened?

The caller threw the exception into the delegator, but the delegator was suspended at `yield from`.

Therefore Python forwarded the exception to the worker's active suspension point.

The worker handled it and yielded another value, so no exception propagated back to the caller.


# Problem 2 — Why can yielding inside `except` be surprising?

A common first implementation handles a command by yielding directly inside the exception handler.

It works, but it introduces an extra suspension point.

We will first observe the issue, then improve the design.


In [8]:
class IgnoreValue(Exception):
    pass


## Step 1 — A first implementation

When `IgnoreValue` is injected, the coroutine yields a message directly from inside `except`.


In [9]:
def awkward_echo() -> Generator[str | None, str, None]:
    while True:
        try:
            value = yield None
            print(f"accepted: {value}")
        except IgnoreValue:
            yield "ignored"


## Step 2 — Drive the coroutine carefully

After `throw(IgnoreValue())`, the generator is paused at the `yield "ignored"` expression.

The next value sent resumes that specific `yield`; it does not immediately become the value of `value = yield None`.


In [10]:
gen = awkward_echo()
assert next(gen) is None

assert gen.send("A") is None
assert gen.throw(IgnoreValue()) == "ignored"

# This value resumes `yield "ignored"` and is discarded.
assert gen.send("B") is None

# Now the generator is back at its normal receive point.
assert gen.send("C") is None

gen.close()


accepted: A
accepted: C


## Why was `"B"` not processed?

The coroutine had not yet returned to `value = yield None`.

It was still paused at the special `yield "ignored"` inside the exception handler.

Sending `"B"` resumed that temporary yield. Since its result was not assigned anywhere, `"B"` disappeared.


## Step 3 — Improve the implementation

We will keep a single predictable `yield` in the loop and store the next outgoing value in `output`.


In [11]:
def stable_echo(events: list[str]) -> Generator[str | None, str, None]:
    output: str | None = None

    while True:
        try:
            value = yield output
        except IgnoreValue:
            output = "ignored"
        else:
            events.append(value)
            output = None


## Step 4 — Verify the improved behavior

Every call now returns to the same suspension point.


In [12]:
events = []
gen = stable_echo(events)

assert next(gen) is None
assert gen.send("A") is None
assert gen.throw(IgnoreValue()) == "ignored"
assert gen.send("B") is None
assert gen.send("C") is None

assert events == ["A", "B", "C"]

gen.close()
events


['A', 'B', 'C']

## Lesson

A coroutine protocol is easier to reason about when each loop has one main suspension point.

An `output` state variable is often preferable to scattering `yield` expressions across `try`, `except`, and `else` branches.


# Problem 3 — Terminate a subgenerator and capture its return value

A subgenerator can terminate in response to an exception and return a summary.

The delegator can capture that summary because a `yield from` expression evaluates to the subgenerator's return value.


In [13]:
class StopCounting(Exception):
    pass


## Step 1 — Write the counting worker

The worker yields its current count.

When it receives `StopCounting`, it returns a dictionary instead of yielding again.


In [14]:
def counter() -> Generator[int, None, dict[str, object]]:
    count = 0

    while True:
        try:
            yield count
            count += 1
        except StopCounting as exc:
            return {
                "count": count,
                "reason": str(exc),
            }


## Step 2 — Wrap it with a delegator

The result of `yield from counter()` is assigned to `worker_result`.


In [15]:
def counting_service() -> Generator[int, None, dict[str, object]]:
    worker_result = yield from counter()

    return {
        "service": "counting",
        "worker": worker_result,
    }


## Step 3 — Run the service

The final call raises `StopIteration` internally.

Our helper extracts its `.value`.


In [16]:
gen = counting_service()

observed = [next(gen), next(gen), next(gen), next(gen)]
result = throw_and_capture(gen, StopCounting("end of input"))

observed, result


([0, 1, 2, 3],
 {'service': 'counting', 'worker': {'count': 3, 'reason': 'end of input'}})

In [17]:
assert observed == [0, 1, 2, 3]

assert result == {
    "service": "counting",
    "worker": {
        "count": 3,
        "reason": "end of input",
    },
}


## Important distinction

These are different channels:

- `yield value` sends a temporary value to the caller while keeping the generator alive;
- `return value` terminates the generator and stores the value in `StopIteration.value`.

`yield from` bridges the second channel by turning the subgenerator's return value into the value of the `yield from` expression.


# Problem 4 — Let the delegator recover from a worker failure

Suppose the worker does not know how to handle a particular failure.

The exception then propagates out of the worker and reaches the delegator.

The delegator will catch it, report a transition, and switch to a fallback worker.


## Step 1 — Create a primary worker that can fail


In [18]:
def primary_worker() -> Generator[str, str, None]:
    value = yield "primary ready"

    while True:
        if value == "FAIL":
            raise ValueError("primary worker rejected the message")

        value = yield f"primary processed {value}"


## Step 2 — Create a fallback worker


In [19]:
def fallback_worker() -> Generator[str, str, None]:
    value = yield "fallback ready"

    while True:
        value = yield f"fallback processed {value}"


## Step 3 — Catch the failure around `yield from`

When `primary_worker` raises `ValueError`, the `yield from` expression also raises `ValueError`.

The delegator can catch it normally.


In [20]:
def failover_service(events: list[str]) -> Generator[str, str, None]:
    try:
        yield from primary_worker()
    except ValueError as exc:
        events.append(f"primary failed: {exc}")
        yield f"switching because: {exc}"
        yield from fallback_worker()


## Step 4 — Observe the transition

The transition message is an ordinary yield by the delegator.

We therefore advance once more to start the fallback worker.


In [21]:
events = []
gen = failover_service(events)

assert next(gen) == "primary ready"
assert gen.send("A") == "primary processed A"
assert gen.send("FAIL") == (
    "switching because: primary worker rejected the message"
)

assert next(gen) == "fallback ready"
assert gen.send("B") == "fallback processed B"

assert events == [
    "primary failed: primary worker rejected the message"
]

gen.close()


## Design observation

The worker owns local processing rules.

The delegator owns service-level policy:

- logging;
- failover;
- retry;
- exception translation;
- lifecycle decisions.

Keeping those concerns separate usually produces cleaner code.


# Problem 5 — Understand `close()` and cleanup order

Calling `close()` injects `GeneratorExit`.

With `yield from`, the active subgenerator is closed as well.

We will verify the exact cleanup order.


## Step 1 — Put cleanup in `finally`

Both generator layers append to the same event log.


In [22]:
def resource_worker(events: list[str]) -> Generator[str, None, None]:
    events.append("worker acquired resource")

    try:
        while True:
            yield "working"
    finally:
        events.append("worker released resource")


def resource_service(events: list[str]) -> Generator[str, None, None]:
    events.append("service started")

    try:
        yield from resource_worker(events)
    finally:
        events.append("service finished")


## Step 2 — Close the outer generator


In [23]:
events = []
gen = resource_service(events)

assert next(gen) == "working"
gen.close()

events


['service started',
 'worker acquired resource',
 'worker released resource',
 'service finished']

In [24]:
assert events == [
    "service started",
    "worker acquired resource",
    "worker released resource",
    "service finished",
]


## Why this order?

The outer service cannot finish its `finally` block until the delegated worker has been closed.

Therefore cleanup proceeds from the deepest active generator outward.


## Step 3 — Demonstrate the forbidden pattern

A generator must not yield after receiving `GeneratorExit`.

Python reports this as `RuntimeError: generator ignored GeneratorExit`.


In [25]:
def invalid_close_handler() -> Generator[str, None, None]:
    try:
        yield "started"
    except GeneratorExit:
        yield "this is illegal"


In [26]:
gen = invalid_close_handler()
assert next(gen) == "started"

close_error = None

try:
    gen.close()
except RuntimeError as exc:
    close_error = str(exc)
finally:
    # Finish cleaning up the demonstration generator.
    try:
        gen.close()
    except RuntimeError:
        pass

assert close_error == "generator ignored GeneratorExit"
close_error


'generator ignored GeneratorExit'

## Best practice

Use `finally` for cleanup.

If you catch `GeneratorExit` explicitly for logging, re-raise it immediately and do not yield.


# Problem 6 — Translate a low-level exception in the delegator

A useful delegator can hide implementation details.

The worker will raise a low-level `ValueError`.

The delegator will translate it into a domain-specific `InvalidCommandError` while preserving the original exception as the cause.


In [27]:
class InvalidCommandError(Exception):
    pass


## Step 1 — Create a strict command parser


In [28]:
def command_parser() -> Generator[str, str, None]:
    command = yield "parser ready"

    while True:
        if not command.startswith("RUN "):
            raise ValueError(f"unsupported command: {command!r}")

        payload = command.removeprefix("RUN ")
        command = yield f"running {payload}"


## Step 2 — Translate the exception with explicit chaining

`raise ... from exc` preserves the causal relationship.


In [29]:
def command_service() -> Generator[str, str, None]:
    try:
        yield from command_parser()
    except ValueError as exc:
        raise InvalidCommandError(
            "The command service received an invalid command."
        ) from exc


## Step 3 — Inspect both exception layers


In [30]:
gen = command_service()
assert next(gen) == "parser ready"
assert gen.send("RUN backup") == "running backup"

caught = None

try:
    gen.send("STOP")
except InvalidCommandError as exc:
    caught = exc

assert caught is not None
assert str(caught) == (
    "The command service received an invalid command."
)
assert isinstance(caught.__cause__, ValueError)
assert str(caught.__cause__) == "unsupported command: 'STOP'"

str(caught), str(caught.__cause__)


('The command service received an invalid command.',
 "unsupported command: 'STOP'")

## Why translate exceptions?

The caller should often depend on domain concepts rather than implementation details.

Exception chaining keeps debugging information without forcing callers to understand the worker's internal exception vocabulary.


# Problem 7 — Build a checkpointable running-statistics coroutine

We will now create a more realistic stateful coroutine.

It receives numeric samples and maintains:

- count;
- total;
- mean;
- minimum;
- maximum.

A `Checkpoint` exception stores a labeled snapshot without terminating the coroutine.


In [31]:
class Checkpoint(Exception):
    def __init__(self, label: str):
        super().__init__(label)
        self.label = label


class ResetStatistics(Exception):
    pass


class FinishStatistics(Exception):
    pass


## Step 1 — Represent state explicitly

A dataclass makes the protocol state easier to inspect and test.


In [32]:
@dataclass
class Statistics:
    count: int = 0
    total: float = 0.0
    minimum: float | None = None
    maximum: float | None = None

    @property
    def mean(self) -> float | None:
        if self.count == 0:
            return None
        return self.total / self.count

    def add(self, value: Real) -> None:
        number = float(value)
        self.count += 1
        self.total += number
        self.minimum = (
            number if self.minimum is None
            else min(self.minimum, number)
        )
        self.maximum = (
            number if self.maximum is None
            else max(self.maximum, number)
        )

    def as_dict(self) -> dict[str, object]:
        return {
            "count": self.count,
            "total": self.total,
            "mean": self.mean,
            "minimum": self.minimum,
            "maximum": self.maximum,
        }


## Step 2 — Implement the coroutine

As in the improved echo example, the loop has one predictable `yield`.

The outgoing response is stored in `output`.


In [33]:
def statistics_worker(
    checkpoints: list[dict[str, object]],
) -> Generator[dict[str, object], Real, dict[str, object]]:
    state = Statistics()
    output: dict[str, object] = {
        "event": "ready",
        "statistics": state.as_dict(),
    }

    while True:
        try:
            value = yield output
        except Checkpoint as command:
            snapshot = {
                "label": command.label,
                **state.as_dict(),
            }
            checkpoints.append(snapshot)
            output = {
                "event": "checkpoint",
                "snapshot": snapshot.copy(),
            }
        except ResetStatistics:
            state = Statistics()
            output = {
                "event": "reset",
                "statistics": state.as_dict(),
            }
        except FinishStatistics as exc:
            return {
                "reason": str(exc),
                "statistics": state.as_dict(),
                "checkpoint_count": len(checkpoints),
            }
        else:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError(
                    "statistics_worker accepts real numbers, excluding bool"
                )

            state.add(value)
            output = {
                "event": "sample",
                "statistics": state.as_dict(),
            }


## Step 3 — Delegate from a service layer


In [34]:
def statistics_service(
    checkpoints: list[dict[str, object]],
) -> Generator[dict[str, object], Real, dict[str, object]]:
    result = yield from statistics_worker(checkpoints)

    return {
        "service": "statistics",
        "status": "finished",
        "result": result,
    }


## Step 4 — Add samples and create a checkpoint


In [35]:
checkpoints: list[dict[str, object]] = []
gen = statistics_service(checkpoints)

ready = next(gen)
sample_1 = gen.send(10)
sample_2 = gen.send(20)
checkpoint = gen.throw(Checkpoint("before reset"))

ready, sample_1, sample_2, checkpoint


({'event': 'ready',
  'statistics': {'count': 0,
   'total': 0.0,
   'mean': None,
   'minimum': None,
   'maximum': None}},
 {'event': 'sample',
  'statistics': {'count': 1,
   'total': 10.0,
   'mean': 10.0,
   'minimum': 10.0,
   'maximum': 10.0}},
 {'event': 'sample',
  'statistics': {'count': 2,
   'total': 30.0,
   'mean': 15.0,
   'minimum': 10.0,
   'maximum': 20.0}},
 {'event': 'checkpoint',
  'snapshot': {'label': 'before reset',
   'count': 2,
   'total': 30.0,
   'mean': 15.0,
   'minimum': 10.0,
   'maximum': 20.0}})

In [36]:
assert ready["statistics"]["count"] == 0
assert sample_1["statistics"]["mean"] == 10.0
assert sample_2["statistics"]["mean"] == 15.0

assert checkpoint["snapshot"] == {
    "label": "before reset",
    "count": 2,
    "total": 30.0,
    "mean": 15.0,
    "minimum": 10.0,
    "maximum": 20.0,
}


## Step 5 — Reset and continue

Resetting does not terminate the service.


In [37]:
reset_response = gen.throw(ResetStatistics())
after_reset_1 = gen.send(-5)
after_reset_2 = gen.send(5)

assert reset_response["statistics"]["count"] == 0
assert after_reset_1["statistics"]["mean"] == -5.0
assert after_reset_2["statistics"]["mean"] == 0.0


## Step 6 — Terminate and capture the final result


In [38]:
final_result = throw_and_capture(
    gen,
    FinishStatistics("normal completion"),
)

final_result


{'service': 'statistics',
 'status': 'finished',
 'result': {'reason': 'normal completion',
  'statistics': {'count': 2,
   'total': 0.0,
   'mean': 0.0,
   'minimum': -5.0,
   'maximum': 5.0},
  'checkpoint_count': 1}}

In [39]:
assert final_result == {
    "service": "statistics",
    "status": "finished",
    "result": {
        "reason": "normal completion",
        "statistics": {
            "count": 2,
            "total": 0.0,
            "mean": 0.0,
            "minimum": -5.0,
            "maximum": 5.0,
        },
        "checkpoint_count": 1,
    },
}

assert checkpoints == [
    {
        "label": "before reset",
        "count": 2,
        "total": 30.0,
        "mean": 15.0,
        "minimum": 10.0,
        "maximum": 20.0,
    }
]


## What this problem combines

- values sent inward with `send()`;
- recoverable commands sent inward with `throw()`;
- a terminating command;
- structured return values;
- `yield from` result capture;
- state reset without generator replacement.


# Problem 8 — Route exceptions through three generator layers

We will create:

```text
outer_service
    -> middle_service
        -> leaf_worker
```

Different exceptions will be handled at different levels.


In [40]:
class LeafAudit(Exception):
    def __init__(self, label: str):
        super().__init__(label)
        self.label = label


class StopMiddle(Exception):
    pass


## Step 1 — The leaf handles audits locally

`LeafAudit` never leaves the leaf worker.


In [41]:
def leaf_worker() -> Generator[str, str, None]:
    output = "leaf ready"

    while True:
        try:
            value = yield output
        except LeafAudit as command:
            output = f"leaf audit: {command.label}"
        else:
            output = f"leaf processed: {value}"


## Step 2 — The middle layer handles shutdown

The leaf does not catch `StopMiddle`.

It propagates to the middle layer, which converts it into a return value.


In [42]:
def middle_service() -> Generator[str, str, dict[str, str]]:
    try:
        yield from leaf_worker()
    except StopMiddle as exc:
        return {
            "middle_status": "stopped",
            "reason": str(exc),
        }


## Step 3 — The outer layer wraps the middle result


In [43]:
def outer_service() -> Generator[str, str, dict[str, object]]:
    middle_result = yield from middle_service()

    return {
        "outer_status": "complete",
        "middle": middle_result,
    }


## Step 4 — Exercise both exception routes


In [44]:
gen = outer_service()

assert next(gen) == "leaf ready"
assert gen.send("A") == "leaf processed: A"

# Handled by the deepest generator.
assert gen.throw(LeafAudit("check-1")) == "leaf audit: check-1"

assert gen.send("B") == "leaf processed: B"

# Not handled by the leaf; handled by the middle.
result = throw_and_capture(
    gen,
    StopMiddle("planned maintenance"),
)

result


{'outer_status': 'complete',
 'middle': {'middle_status': 'stopped', 'reason': 'planned maintenance'}}

In [45]:
assert result == {
    "outer_status": "complete",
    "middle": {
        "middle_status": "stopped",
        "reason": "planned maintenance",
    },
}


## Routing rule

An injected exception is offered first to the deepest active generator.

If that generator does not handle it, normal stack unwinding continues through each delegator until a matching handler is found.


# Problem 9 — Build a restartable supervisor

A supervisor delegates to a worker.

The worker owns its local state.

The supervisor owns restart policy.


In [46]:
class QueryWorker(Exception):
    pass


class RestartWorker(Exception):
    pass


class StopSupervisor(Exception):
    pass


## Step 1 — Create a worker with local state

`QueryWorker` is handled locally.

`RestartWorker` and `StopSupervisor` are deliberately not handled.


In [47]:
def supervised_worker(
    worker_id: int,
) -> Generator[dict[str, object], int, None]:
    total = 0
    output: dict[str, object] = {
        "event": "ready",
        "worker_id": worker_id,
        "total": total,
    }

    while True:
        try:
            value = yield output
        except QueryWorker:
            output = {
                "event": "state",
                "worker_id": worker_id,
                "total": total,
            }
        else:
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(
                    "supervised_worker accepts integers, excluding bool"
                )

            total += value
            output = {
                "event": "updated",
                "worker_id": worker_id,
                "total": total,
            }


## Step 2 — Create the supervisor loop

When the worker receives `RestartWorker`, it has no handler, so the exception reaches the supervisor.

The supervisor catches it, increments the worker id, and delegates to a fresh worker.


In [48]:
def supervisor() -> Generator[
    dict[str, object],
    int,
    dict[str, object],
]:
    worker_id = 0
    restart_count = 0

    while True:
        try:
            yield from supervised_worker(worker_id)
        except RestartWorker:
            restart_count += 1
            worker_id += 1
            continue
        except StopSupervisor as exc:
            return {
                "status": "stopped",
                "reason": str(exc),
                "restart_count": restart_count,
                "last_worker_id": worker_id,
            }


## Step 3 — Observe a restart

The same `throw(RestartWorker())` call enters the old worker, propagates to the supervisor, creates a new worker, and returns that new worker's first yielded value.


In [49]:
gen = supervisor()

assert next(gen) == {
    "event": "ready",
    "worker_id": 0,
    "total": 0,
}

assert gen.send(5)["total"] == 5
assert gen.throw(QueryWorker()) == {
    "event": "state",
    "worker_id": 0,
    "total": 5,
}

restarted = gen.throw(RestartWorker())

assert restarted == {
    "event": "ready",
    "worker_id": 1,
    "total": 0,
}

assert gen.send(3)["total"] == 3


## Step 4 — Stop the supervisor and capture its report


In [50]:
report = throw_and_capture(
    gen,
    StopSupervisor("deployment"),
)

assert report == {
    "status": "stopped",
    "reason": "deployment",
    "restart_count": 1,
    "last_worker_id": 1,
}

report


{'status': 'stopped',
 'reason': 'deployment',
 'restart_count': 1,
 'last_worker_id': 1}

## Why this separation is valuable

The worker does not know whether failures should cause:

- a restart;
- a fallback;
- a permanent shutdown;
- an alert;
- escalation to another layer.

Those decisions belong to the supervisor.


# Problem 10 — Capstone: transactional batch processing

We will build a generator-based batch processor.

Ordinary values are staged.

Exception commands control the batch:

- `CommitBatch`: apply all pending values;
- `RollbackBatch`: discard pending values;
- `InspectBatch`: return the current pending values;
- `AbortBatch`: terminate and return a final report.

The worker will be delegated through a service layer.


In [51]:
class CommitBatch(Exception):
    pass


class RollbackBatch(Exception):
    pass


class InspectBatch(Exception):
    pass


class AbortBatch(Exception):
    pass


## Step 1 — Decide on the response protocol

Every operation will yield a dictionary with an `"event"` field.

This makes the protocol easier to extend and test than unrelated strings or print statements.


## Step 2 — Implement the worker

The callback `apply_batch` represents the external operation performed during a commit.

In production, this callback could fail; robust code would need idempotency and partial-failure handling.


In [52]:
def batch_worker(
    apply_batch: Callable[[list[object]], None],
) -> Generator[
    dict[str, object],
    object,
    dict[str, object],
]:
    pending: list[object] = []
    committed_count = 0

    output: dict[str, object] = {
        "event": "ready",
        "pending": [],
        "committed_count": committed_count,
    }

    while True:
        try:
            item = yield output
        except InspectBatch:
            output = {
                "event": "inspection",
                "pending": pending.copy(),
                "committed_count": committed_count,
            }
        except CommitBatch:
            batch = pending.copy()
            apply_batch(batch)
            committed_count += len(batch)
            pending.clear()

            output = {
                "event": "committed",
                "batch": batch,
                "pending": [],
                "committed_count": committed_count,
            }
        except RollbackBatch:
            discarded = pending.copy()
            pending.clear()

            output = {
                "event": "rolled back",
                "discarded": discarded,
                "pending": [],
                "committed_count": committed_count,
            }
        except AbortBatch as exc:
            return {
                "reason": str(exc),
                "discarded_on_abort": pending.copy(),
                "committed_count": committed_count,
            }
        else:
            pending.append(item)

            output = {
                "event": "staged",
                "item": item,
                "pending": pending.copy(),
                "committed_count": committed_count,
            }


## Step 3 — Add a service layer

The service captures the worker's return value and adds service-level metadata.


In [53]:
def batch_service(
    apply_batch: Callable[[list[object]], None],
) -> Generator[
    dict[str, object],
    object,
    dict[str, object],
]:
    worker_result = yield from batch_worker(apply_batch)

    return {
        "service": "batch-service",
        "status": "aborted",
        "worker_result": worker_result,
    }


## Step 4 — Stage and inspect values


In [54]:
applied: list[object] = []

def apply_batch(batch: list[object]) -> None:
    applied.extend(batch)


gen = batch_service(apply_batch)

assert next(gen)["event"] == "ready"
assert gen.send("A")["pending"] == ["A"]
assert gen.send("B")["pending"] == ["A", "B"]

inspection = gen.throw(InspectBatch())

assert inspection == {
    "event": "inspection",
    "pending": ["A", "B"],
    "committed_count": 0,
}


## Step 5 — Commit the first batch


In [55]:
committed = gen.throw(CommitBatch())

assert committed == {
    "event": "committed",
    "batch": ["A", "B"],
    "pending": [],
    "committed_count": 2,
}

assert applied == ["A", "B"]


## Step 6 — Stage another batch and roll it back


In [56]:
assert gen.send("C")["pending"] == ["C"]
assert gen.send("D")["pending"] == ["C", "D"]

rolled_back = gen.throw(RollbackBatch())

assert rolled_back == {
    "event": "rolled back",
    "discarded": ["C", "D"],
    "pending": [],
    "committed_count": 2,
}

assert applied == ["A", "B"]


## Step 7 — Abort with pending work

Pending values are reported as discarded in the final result.


In [57]:
assert gen.send("E")["pending"] == ["E"]

final_report = throw_and_capture(
    gen,
    AbortBatch("client cancelled"),
)

final_report


{'service': 'batch-service',
 'status': 'aborted',
 'worker_result': {'reason': 'client cancelled',
  'discarded_on_abort': ['E'],
  'committed_count': 2}}

In [58]:
assert final_report == {
    "service": "batch-service",
    "status": "aborted",
    "worker_result": {
        "reason": "client cancelled",
        "discarded_on_abort": ["E"],
        "committed_count": 2,
    },
}

assert applied == ["A", "B"]


## Capstone discussion

This design demonstrates a complete bidirectional protocol:

- ordinary data travels inward with `send()`;
- control signals travel inward with `throw()`;
- responses travel outward with `yield`;
- the final report travels outward through `return` and `StopIteration.value`;
- `yield from` connects the worker protocol to the service protocol.

It is educational and can be useful in synchronous pipelines.

For asynchronous I/O or concurrent tasks, modern `async`/`await` code is usually clearer.


# Review questions

Try answering these without running code.

1. Where is an exception from `delegator.throw(E())` raised first?
2. What happens if the active subgenerator catches the exception and yields a value?
3. What happens if it does not catch the exception?
4. How does a caller retrieve a generator's return value?
5. Why is yielding after `GeneratorExit` illegal?
6. Why can `yield` inside `except` create a discarded `send()` value?
7. What does the `yield from` expression evaluate to when the subgenerator returns?
8. Which layer should own retry or failover policy?


# Review answers

1. It is first offered to the deepest active subgenerator.
2. The yielded value becomes the result of the caller's `throw()` call, and the generator remains alive.
3. It propagates outward through the delegator chain until a handler is found or the caller receives it.
4. Catch `StopIteration` and read `.value`; a delegator can capture it directly as the value of `yield from`.
5. Closing requires termination; yielding contradicts that protocol and produces `RuntimeError`.
6. The generator pauses at that special yield, so the next `send()` resumes it rather than the normal receive expression.
7. The subgenerator's return value.
8. Usually the delegator or supervisor layer.


# Additional advanced exercises

## Exercise A — Retry budget

Extend the supervisor so that `RestartWorker` succeeds only three times. On the fourth restart request, terminate and return a report.

## Exercise B — Snapshot persistence failure

Modify the statistics worker so its checkpoint sink can raise an exception. Decide whether the worker, delegator, or caller should handle it.

## Exercise C — Nested cleanup

Add `finally` blocks to the three-layer routing example and record the cleanup order after `close()`.

## Exercise D — Exception hierarchy

Create a common base class for protocol exceptions and use separate subclasses for recoverable, terminating, and administrative commands.

## Exercise E — Typed messages instead of `throw()`

Rebuild the batch processor using dataclass command objects sent through `send()`. Compare readability and type-checking.

## Exercise F — Manual delegation

Implement a simplified forwarding loop that approximates `yield from` for `send()`, `throw()`, `close()`, and return values. Use it only as a learning exercise.


# Common mistakes

- Sending a non-`None` value before priming the generator.
- Expecting `close()` to return the generator's return value.
- Catching `BaseException` and accidentally swallowing `GeneratorExit`.
- Yielding after `GeneratorExit`.
- Forgetting that the deepest active subgenerator receives `throw()` first.
- Adding hidden suspension points by yielding directly in exception handlers.
- Returning only printed messages instead of structured state.
- Using exceptions for routine commands when typed messages would be simpler.
- Reimplementing `yield from` in production code.


# Final summary

`yield from` is a protocol bridge, not merely a shorter `for` loop.

It coordinates:

- outward yielded values;
- inward sent values;
- inward exceptions;
- closing;
- subgenerator return values.

The most maintainable generator protocols make state explicit, keep suspension points predictable, use narrow exception types, place cleanup in `finally`, and verify every transition with assertions.
